In [ ]:
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from dotenv import load_dotenv
from pathlib import Path
import numpy as np
from typing import List, Dict, Any, Optional, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [37]:
### Read all the text files inside the directory
def process_all_texts(text_directory):
    """Process all .txt files in a directory"""
    all_documents = []
    text_dir = Path(text_directory)
    
    # Find all text files recursively
    text_files = list(text_dir.glob("**/*.txt"))
    
    print(f"Found {len(text_files)} text files to process")
    
    for text_file in text_files:
        print(f"\nProcessing: {text_file.name}")
        try:
            loader = TextLoader(str(text_file), encoding="utf-8")
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = text_file.name
                doc.metadata['file_type'] = 'txt'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} documents")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all text files in the data directory
all_text_documents = process_all_texts("../data")

Found 156 text files to process

Processing: Email_20220320_005.txt
  ✓ Loaded 1 documents

Processing: Email_20220408_021.txt
  ✓ Loaded 1 documents

Processing: Email_20220619_019.txt
  ✓ Loaded 1 documents

Processing: Email_20221118_002.txt
  ✓ Loaded 1 documents

Processing: Email_20221120_016.txt
  ✓ Loaded 1 documents

Processing: Email_20221209_023.txt
  ✓ Loaded 1 documents

Processing: Email_20221225_015.txt
  ✓ Loaded 1 documents

Processing: Email_20230118_011.txt
  ✓ Loaded 1 documents

Processing: Email_20230314_024.txt
  ✓ Loaded 1 documents

Processing: Email_20230422_027.txt
  ✓ Loaded 1 documents

Processing: Email_20230424_004.txt
  ✓ Loaded 1 documents

Processing: Email_20230424_007.txt
  ✓ Loaded 1 documents

Processing: Email_20230504_022.txt
  ✓ Loaded 1 documents

Processing: Email_20230725_025.txt
  ✓ Loaded 1 documents

Processing: Email_20230814_029.txt
  ✓ Loaded 1 documents

Processing: Email_20230831_028.txt
  ✓ Loaded 1 documents

Processing: Email_20230

In [38]:
all_text_documents

[Document(metadata={'source': '..\\data\\emails\\Email_20220320_005.txt', 'source_file': 'Email_20220320_005.txt', 'file_type': 'txt'}, page_content='\nFrom: pm@controlcutter.no\nTo: client@equinorasa.com\nDate: 2022-03-20 00:00\nSubject: Equipment Issue Report - Forties Field\n\nDear Sarah,\n\nReporting minor hydraulic pressure fluctuation on HC-1890-06\n\nNo impact to project timeline expected\n\nLooking forward to your feedback.\n\nBest regards,\nTom Hansen\nControl Cutter AS\n'),
 Document(metadata={'source': '..\\data\\emails\\Email_20220408_021.txt', 'source_file': 'Email_20220408_021.txt', 'file_type': 'txt'}, page_content='\nFrom: pm@controlcutter.no\nTo: client@bpnorthsea.com\nDate: 2022-04-08 00:00\nSubject: Project Status Update - Brent Field\n\nDear John,\n\nWeekly project status update for CC-2022-106\n\nCurrent progress: {progress}% complete\n\nLet me know if you have questions.\n\nBest regards,\nLars Eriksen\nControl Cutter AS\n'),
 Document(metadata={'source': '..\\data

In [39]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


In [40]:
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


In [41]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "text_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "text document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: text_documents
Existing documents in collection: 424


In [42]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [43]:
chunks=split_documents(all_text_documents)
chunks

Split 156 documents into 424 chunks

Example chunk:
Content: From: pm@controlcutter.no
To: client@equinorasa.com
Date: 2022-03-20 00:00
Subject: Equipment Issue Report - Forties Field

Dear Sarah,

Reporting minor hydraulic pressure fluctuation on HC-1890-06

N...
Metadata: {'source': '..\\data\\emails\\Email_20220320_005.txt', 'source_file': 'Email_20220320_005.txt', 'file_type': 'txt'}


[Document(metadata={'source': '..\\data\\emails\\Email_20220320_005.txt', 'source_file': 'Email_20220320_005.txt', 'file_type': 'txt'}, page_content='From: pm@controlcutter.no\nTo: client@equinorasa.com\nDate: 2022-03-20 00:00\nSubject: Equipment Issue Report - Forties Field\n\nDear Sarah,\n\nReporting minor hydraulic pressure fluctuation on HC-1890-06\n\nNo impact to project timeline expected\n\nLooking forward to your feedback.\n\nBest regards,\nTom Hansen\nControl Cutter AS'),
 Document(metadata={'source': '..\\data\\emails\\Email_20220408_021.txt', 'source_file': 'Email_20220408_021.txt', 'file_type': 'txt'}, page_content='From: pm@controlcutter.no\nTo: client@bpnorthsea.com\nDate: 2022-04-08 00:00\nSubject: Project Status Update - Brent Field\n\nDear John,\n\nWeekly project status update for CC-2022-106\n\nCurrent progress: {progress}% complete\n\nLet me know if you have questions.\n\nBest regards,\nLars Eriksen\nControl Cutter AS'),
 Document(metadata={'source': '..\\data\\emails

In [44]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

##store int the vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 424 texts...


Batches: 100%|██████████| 14/14 [00:12<00:00,  1.13it/s]


Generated embeddings with shape: (424, 384)
Adding 424 documents to vector store...
Successfully added 424 documents to vector store
Total documents in collection: 848


In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore,embedding_manager)

In [46]:
rag_retriever

In [54]:
rag_retriever.retrieve("EQUIPMENT OVERVIEW")

Retrieving documents for query: 'EQUIPMENT OVERVIEW'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.92it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_01180505_422',
  'content': 'Safety Equipment Required:\n- Cut-resistant gloves (EN 388 Level 5)\n- Safety glasses with side shields\n- Hard hat with chin strap (offshore operations)\n- Personal flotation device (when on vessel)\n- Two-way communication device\n\nEmergency Procedures:\n- Hydraulic leak: Shut down system, evacuate area, alert supervisor\n- Equipment malfunction: Emergency stop, secure equipment, report\n- Weather deterioration: Recover equipment, postpone operations\n- Injury: First aid, medical evacuation if needed\n\nSECTION 6: HANDS-ON DEMONSTRATION (30 minutes)\n- Equipment walkthrough (actual HC-2000 unit)\n- Blade inspection demonstration\n- Control system operation\n- Pre-deployment checklist practice\n\nASSESSMENT:\n- Written quiz (20 questions, 80% pass required)\n- Practical demonstration (pre-deployment check)\n\nNEXT STEPS:\nUpon completion:\n→ Module 002: Advanced Equipment Operation\n→ Module 003: Troubleshooting and Maintenance\n→ On-the-job 

In [61]:
rag_retriever.retrieve("equipment overview")

Retrieving documents for query: 'equipment overview'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 24.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_01180505_422',
  'content': 'Safety Equipment Required:\n- Cut-resistant gloves (EN 388 Level 5)\n- Safety glasses with side shields\n- Hard hat with chin strap (offshore operations)\n- Personal flotation device (when on vessel)\n- Two-way communication device\n\nEmergency Procedures:\n- Hydraulic leak: Shut down system, evacuate area, alert supervisor\n- Equipment malfunction: Emergency stop, secure equipment, report\n- Weather deterioration: Recover equipment, postpone operations\n- Injury: First aid, medical evacuation if needed\n\nSECTION 6: HANDS-ON DEMONSTRATION (30 minutes)\n- Equipment walkthrough (actual HC-2000 unit)\n- Blade inspection demonstration\n- Control system operation\n- Pre-deployment checklist practice\n\nASSESSMENT:\n- Written quiz (20 questions, 80% pass required)\n- Practical demonstration (pre-deployment check)\n\nNEXT STEPS:\nUpon completion:\n→ Module 002: Advanced Equipment Operation\n→ Module 003: Troubleshooting and Maintenance\n→ On-the-job 

RAG Pipeline- VectorDB To LLM Output Generation

In [62]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
import os
from dotenv import load_dotenv
load_dotenv()
print(os.getenv("GROQ_API_KEY"))

In [ ]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq
        (
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [ ]:
# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key = os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Please set your GROQ_API_KEY environment variable to use the LLM.


In [ ]:


# Always load .env from project root, even when running inside /notebook
root_dir = Path.cwd()
if (root_dir / ".env").exists():
    env_path = root_dir / ".env"
else:
    env_path = root_dir.parent / ".env"

load_dotenv(dotenv_path=env_path)

print("Loaded .env from:", env_path)
print("GROQ_API_KEY loaded:", bool(os.getenv("GROQ_API_KEY")))


Loaded .env from: c:\Users\ajaos\Desktop\2025 PROJECT\spliichx-projects-repositories\rag-proj\.env
GROQ_API_KEY loaded: False
